# Pandas 시작하기: Series, DataFrame, 결측값 처리, apply, 정렬, 기술통계, 상관관계·공분산 실습

## 실습 목표

실습은 두 종류의 데이터를 사용합니다.

1. **직접 만든 학습자/수강 데이터 DataFrame**  
   - Series와 DataFrame 생성
   - index/columns/values/dtypes/shape 확인
   - 결측값 처리
   - `loc`, `iloc`, 컬럼 추가·삭제
   - `apply`, 정렬, 순위, 유일값 처리

2. **기존 실제 데이터 `boston.csv`**  
   - CSV 로딩
   - 기술통계
   - 변수 간 상관관계와 공분산
   - `corr`, `cov`, `corrwith` 활용

> 참고: `boston.csv`는 교육용 통계 실습 데이터로 사용합니다. 이 노트북에서는 데이터 분석 문법과 통계 계산 방법을 익히는 데 초점을 둡니다.

## 0. 실습 환경 준비

먼저 사용할 라이브러리를 불러옵니다.

In [ ]:
!pip install matplotlib

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

DATA_DIR = Path("./data") 

print("pandas version:", pd.__version__)
print("data directory:", DATA_DIR)

pandas version: 3.0.3
data directory: data


---

# Part 1. Pandas 시작하기

Pandas는 표 형태의 데이터를 다루기 위한 대표적인 파이썬 라이브러리입니다.  
핵심 자료구조는 다음 두 가지입니다.

- **Series**: 1차원 라벨 배열
- **DataFrame**: 2차원 표 형태 자료구조

## 1. Series 생성과 기본 속성

Series는 값과 인덱스를 함께 가지는 1차원 자료구조입니다.

In [4]:
scores = pd.Series([88, 92, 75, 64, 95], index=["민수", "지영", "현우", "수진", "하린"], name="중간점수")
scores

민수    88
지영    92
현우    75
수진    64
하린    95
Name: 중간점수, dtype: int64

In [5]:
print("값(values):", scores.values)
print("인덱스(index):", scores.index)
print("자료형(dtype):", scores.dtype)
print("Series 이름(name):", scores.name)

scores.index.name = "이름"
print("index name:", scores.index.name)
scores

값(values): [88 92 75 64 95]
인덱스(index): Index(['민수', '지영', '현우', '수진', '하린'], dtype='str')
자료형(dtype): int64
Series 이름(name): 중간점수
index name: 이름


이름
민수    88
지영    92
현우    75
수진    64
하린    95
Name: 중간점수, dtype: int64

### Series 인덱싱과 연산

Series는 인덱스 이름으로 접근할 수 있고, NumPy 배열처럼 벡터화 연산도 가능합니다.

In [6]:
print("지영 점수:", scores["지영"])
print("80점 이상 여부")
display(scores >= 80)

bonus_scores = scores + 3
bonus_scores.name = "가산점반영점수"
bonus_scores

지영 점수: 92
80점 이상 여부


이름
민수     True
지영     True
현우    False
수진    False
하린     True
Name: 중간점수, dtype: bool

이름
민수    91
지영    95
현우    78
수진    67
하린    98
Name: 가산점반영점수, dtype: int64

### 실습 1

아래 셀을 실행한 뒤, 직접 값을 바꿔 보세요.

In [7]:
# 실습: 특정 학생의 점수를 선택하고, 평균 이상인 학생만 필터링해 봅니다.
mean_score = scores.mean()
print("평균:", mean_score)

scores[scores >= mean_score]

평균: 82.8


이름
민수    88
지영    92
하린    95
Name: 중간점수, dtype: int64

---

# Part 2. 직접 만든 DataFrame으로 Pandas 기본기 실습

강의자료의 DataFrame 기본 개념을 다루기 위해 학습자 수강 데이터를 직접 만들어 봅니다.  
이 데이터에는 일부러 결측값을 포함했습니다.

## 2. DataFrame 생성

딕셔너리, 리스트, NumPy 배열 등 다양한 형태의 데이터를 DataFrame으로 만들 수 있습니다.

In [8]:
student_data = {
    "이름": ["민수", "지영", "현우", "수진", "하린", "도윤", "서연", "태오"],
    "반": ["A", "A", "B", "B", "A", "B", "A", "B"],
    "도시": ["서울", "부산", "서울", "대전", "부산", "서울", np.nan, "대전"],
    "중간점수": [88, 92, 75, np.nan, 95, 70, 84, 63],
    "기말점수": [91, 87, 79, 85, np.nan, 72, 88, 68],
    "출석률": [0.95, 0.90, 0.85, 0.80, 0.98, np.nan, 0.92, 0.76],
    "프로젝트점수": [90, 85, np.nan, 88, 93, 70, 86, 65],
    "수료여부": [True, True, True, True, True, False, True, False],
}

students = pd.DataFrame(student_data)
students

,이름,반,도시,중간점수,기말점수,출석률,프로젝트점수,수료여부
0,민수,A,서울,88.0,91.0,0.95,90.0,True
1,지영,A,부산,92.0,87.0,0.90,85.0,True
2,현우,B,서울,75.0,79.0,0.85,NaN,True
3,수진,B,대전,NaN,85.0,0.80,88.0,True
4,하린,A,부산,95.0,NaN,0.98,93.0,True
5,도윤,B,서울,70.0,72.0,NaN,70.0,False
6,서연,A,NaN,84.0,88.0,0.92,86.0,True
7,태오,B,대전,63.0,68.0,0.76,65.0,False


## 3. DataFrame 주요 속성 확인

DataFrame은 값(`values`), 행 인덱스(`index`), 열 이름(`columns`), 자료형(`dtypes`), 크기(`shape`) 등의 속성을 가집니다.

In [9]:
print("index:", students.index)
print("columns:", students.columns.tolist())
print("shape:", students.shape)
print("size:", students.size)
print("ndim:", students.ndim)

students.dtypes

index: RangeIndex(start=0, stop=8, step=1)
columns: ['이름', '반', '도시', '중간점수', '기말점수', '출석률', '프로젝트점수', '수료여부']
shape: (8, 8)
size: 64
ndim: 2


이름            str
반             str
도시            str
중간점수      float64
기말점수      float64
출석률       float64
프로젝트점수    float64
수료여부         bool
dtype: object

In [10]:
# values는 실제 데이터 배열을 반환합니다.
students.values[:3]

array([['민수', 'A', '서울', 88.0, 91.0, 0.95, 90.0, True],
       ['지영', 'A', '부산', 92.0, 87.0, 0.9, 85.0, True],
       ['현우', 'B', '서울', 75.0, 79.0, 0.85, nan, True]], dtype=object)

### index와 columns 이름 지정

강의자료처럼 DataFrame의 행 방향 인덱스와 열 방향 인덱스에 이름을 붙일 수 있습니다.

In [11]:
students.index.name = "row_id"
students.columns.name = "columns"
students

columns,이름,반,도시,중간점수,기말점수,출석률,프로젝트점수,수료여부
row_id,,,,,,,,
0,민수,A,서울,88.0,91.0,0.95,90.0,True
1,지영,A,부산,92.0,87.0,0.90,85.0,True
2,현우,B,서울,75.0,79.0,0.85,NaN,True
3,수진,B,대전,NaN,85.0,0.80,88.0,True
4,하린,A,부산,95.0,NaN,0.98,93.0,True
5,도윤,B,서울,70.0,72.0,NaN,70.0,False
6,서연,A,NaN,84.0,88.0,0.92,86.0,True
7,태오,B,대전,63.0,68.0,0.76,65.0,False


## 4. 컬럼 선택, 여러 컬럼 선택, 컬럼 추가

DataFrame에서 하나의 열을 선택하면 Series가 반환됩니다.  
여러 열을 선택하면 DataFrame이 반환됩니다.

In [12]:
# 하나의 컬럼 선택: Series
name_series = students["이름"]
print(type(name_series))
display(name_series)

# 여러 컬럼 선택: DataFrame
selected_df = students[["이름", "반", "중간점수", "기말점수"]]
print(type(selected_df))
selected_df

<class 'pandas.Series'>


row_id
0    민수
1    지영
2    현우
3    수진
4    하린
5    도윤
6    서연
7    태오
Name: 이름, dtype: str

<class 'pandas.DataFrame'>


columns,이름,반,중간점수,기말점수
row_id,,,,
0,민수,A,88.0,91.0
1,지영,A,92.0,87.0
2,현우,B,75.0,79.0
3,수진,B,NaN,85.0
4,하린,A,95.0,NaN
5,도윤,B,70.0,72.0
6,서연,A,84.0,88.0
7,태오,B,63.0,68.0


In [ ]:
# 기존 컬럼을 활용하여 새로운 컬럼 만들기
students["평균점수"] = students[["중간점수", "기말점수", "프로젝트점수"]].mean(axis=1)
students["우수여부"] = students["평균점수"] >= 85
students

### Series를 이용한 컬럼 입력과 인덱스 정렬

Series를 컬럼에 넣을 때는 Series의 인덱스와 DataFrame의 인덱스가 맞는 위치에 값이 들어갑니다.

In [ ]:
# 일부 행에만 멘토링 시간을 입력합니다.
mentoring_hours = pd.Series([2, 3, 1], index=[1, 4, 6])
students["멘토링시간"] = mentoring_hours
students

## 5. loc와 iloc로 행/열 선택

- `loc`: 실제 인덱스 이름 또는 라벨 기반 선택
- `iloc`: 정수 위치 기반 선택

In [ ]:
# loc: 인덱스 라벨 기준
students.loc[1]

In [ ]:
# loc: 행 범위와 열 이름을 함께 지정
students.loc[1:4, ["이름", "반", "중간점수", "기말점수", "평균점수"]]

In [ ]:
# iloc: 정수 위치 기준
students.iloc[0:3, 0:5]

In [ ]:
# Boolean indexing: 조건이 True인 행만 선택
students[students["평균점수"] >= 85]

## 6. 컬럼 삭제

`del` 또는 `drop`을 이용해 컬럼을 삭제할 수 있습니다.  
실습에서는 원본을 보존하기 위해 복사본에서 삭제합니다.

In [ ]:
students_deleted = students.copy()
del students_deleted["우수여부"]
students_deleted.head()

In [ ]:
# 여러 컬럼 삭제는 drop 사용
students_deleted2 = students.drop(columns=["우수여부", "멘토링시간"])
students_deleted2.head()

---

# Part 3. 결측값 처리

실제 데이터에는 값이 비어 있는 경우가 많습니다.  
Pandas에서는 결측값을 보통 `NaN`으로 표현하고, `isnull`, `notnull`, `dropna`, `fillna` 등을 사용합니다.

## 7. 결측값 확인

In [ ]:
students.isnull()

In [ ]:
# 컬럼별 결측값 개수
students.isnull().sum()

In [ ]:
# 결측값이 하나라도 있는 행만 확인
students[students.isnull().any(axis=1)]

## 8. dropna로 결측값 제거

`dropna()`는 결측값이 있는 행 또는 열을 제거합니다.  
다만 데이터 손실이 발생할 수 있으므로 실제 분석에서는 신중하게 사용합니다.

In [ ]:
# 결측값이 하나라도 있는 행 제거
students_drop_any = students.dropna(how="any")
students_drop_any

In [ ]:
# 모든 값이 결측인 행만 제거: 현재 데이터에서는 제거되는 행이 없을 수 있습니다.
students.dropna(how="all")

## 9. fillna로 결측값 채우기

수치형 컬럼은 평균이나 중앙값으로, 범주형 컬럼은 최빈값 또는 별도 문자열로 채우는 방법을 많이 사용합니다.

In [ ]:
students_filled = students.copy()

score_cols = ["중간점수", "기말점수", "프로젝트점수", "출석률", "멘토링시간"]
for col in score_cols:
    students_filled[col] = students_filled[col].fillna(students_filled[col].mean())

students_filled["도시"] = students_filled["도시"].fillna("미입력")

# 평균점수는 결측값을 채운 뒤 다시 계산합니다.
students_filled["평균점수"] = students_filled[["중간점수", "기말점수", "프로젝트점수"]].mean(axis=1)
students_filled["우수여부"] = students_filled["평균점수"] >= 85

students_filled

In [ ]:
students_filled.isnull().sum()

---

# Part 4. apply와 사용자 정의 함수

`apply`는 행 또는 열 단위로 함수를 적용할 때 사용합니다.

- `axis=0`: 열 단위 적용
- `axis=1`: 행 단위 적용

## 10. 컬럼 단위 apply

In [ ]:
# 각 수치형 컬럼의 범위(max - min)를 계산합니다.
range_func = lambda x: x.max() - x.min()

students_filled[["중간점수", "기말점수", "프로젝트점수", "평균점수"]].apply(range_func, axis=0)

## 11. 행 단위 apply

각 학습자의 평균점수와 출석률을 기준으로 평가 등급을 만들어 봅니다.

In [ ]:
def assign_grade(row):
    if row["평균점수"] >= 90 and row["출석률"] >= 0.9:
        return "A+"
    elif row["평균점수"] >= 85:
        return "A"
    elif row["평균점수"] >= 75:
        return "B"
    else:
        return "C"

students_filled["평가등급"] = students_filled.apply(assign_grade, axis=1)
students_filled[["이름", "평균점수", "출석률", "평가등급"]]

In [ ]:
# lambda 함수로 간단한 컬럼 만들기
students_filled["수료상태"] = students_filled.apply(
    lambda row: "수료" if row["수료여부"] and row["평균점수"] >= 70 else "미수료",
    axis=1,
)
students_filled[["이름", "수료여부", "평균점수", "수료상태"]]

---

# Part 5. 정렬, 순위, 유일값 처리

## 12. sort_index와 sort_values

In [ ]:
# 인덱스 기준 정렬
students_filled.sort_index(ascending=False)

In [ ]:
# 값 기준 정렬: 평균점수가 높은 순서
students_filled.sort_values(by="평균점수", ascending=False)

In [ ]:
# 여러 기준으로 정렬: 반 오름차순, 평균점수 내림차순
students_filled.sort_values(by=["반", "평균점수"], ascending=[True, False])

## 13. rank로 순위 매기기

In [ ]:
students_filled["평균점수순위"] = students_filled["평균점수"].rank(ascending=False, method="min")
students_filled[["이름", "평균점수", "평균점수순위"]].sort_values("평균점수순위")

## 14. unique, value_counts, isin

In [ ]:
print("도시 unique:", students_filled["도시"].unique())
print("반 unique:", students_filled["반"].unique())

students_filled["도시"].value_counts()

In [ ]:
# 서울 또는 부산 학습자만 선택
students_filled[students_filled["도시"].isin(["서울", "부산"])]

---

# Part 6. 기술통계와 요약 통계

`count`, `describe`, `sum`, `mean`, `median`, `var`, `std`, `quantile`, `diff` 등을 실습합니다.

## 15. 직접 만든 DataFrame의 기술통계

In [ ]:
students_filled.describe()

In [ ]:
score_df = students_filled[["중간점수", "기말점수", "프로젝트점수", "평균점수", "출석률"]]

summary = pd.DataFrame({
    "count": score_df.count(),
    "mean": score_df.mean(),
    "median": score_df.median(),
    "std": score_df.std(),
    "var": score_df.var(),
    "min": score_df.min(),
    "max": score_df.max(),
    "q25": score_df.quantile(0.25),
    "q75": score_df.quantile(0.75),
})
summary

In [ ]:
# diff: 이전 행과의 차이. 시계열 데이터에서 특히 자주 사용합니다.
students_filled.sort_values("평균점수", ascending=False)[["이름", "평균점수"]].assign(
    이전순위와점수차=lambda df: df["평균점수"].diff()
)

---

# Part 7. 실제 데이터 boston.csv 로딩과 DataFrame 분석

이번에는 기존 실제 데이터인 `boston.csv`를 사용합니다.  
이 데이터는 수치형 컬럼이 많아 기술통계, 상관관계, 공분산 실습에 적합합니다.

## 16. CSV 파일 읽기

In [ ]:
boston_path = DATA_DIR / "boston.csv"
boston = pd.read_csv(boston_path)

boston.head()

In [ ]:
print("shape:", boston.shape)
print("columns:", boston.columns.tolist())
print("dtypes:")
display(boston.dtypes)

boston.info()

## 17. boston.csv의 기본 DataFrame 조작

실제 데이터에서도 앞에서 배운 Pandas 기본기를 동일하게 사용할 수 있습니다.

In [ ]:
# 하나의 컬럼 선택: Series
price = boston["Price"]
print(type(price))
price.head()

In [ ]:
# 여러 컬럼 선택: DataFrame
boston_small = boston[["Price", "RM", "LSTAT", "PTRATIO", "NOX", "DIS"]]
boston_small.head()

In [ ]:
# loc와 iloc 사용
print("loc 예시")
display(boston.loc[0:4, ["Price", "RM", "LSTAT"]])

print("iloc 예시")
display(boston.iloc[0:5, 0:4])

In [ ]:
# 조건 필터링: 방 개수 RM이 높고 가격 Price도 높은 지역
boston[(boston["RM"] >= boston["RM"].quantile(0.75)) & (boston["Price"] >= boston["Price"].quantile(0.75))].head()

## 18. 실제 데이터의 결측값 확인

In [ ]:
boston.isnull().sum()

`boston.csv` 자체에는 결측값이 없을 수 있습니다.  
결측값 처리 실습을 위해 일부 값을 의도적으로 비워서 연습용 데이터를 만듭니다.

In [ ]:
boston_missing = boston.copy()

# 재현 가능한 실습을 위해 고정된 위치에 결측값을 만듭니다.
boston_missing.loc[[0, 3, 7], "RM"] = np.nan
boston_missing.loc[[2, 5, 9], "LSTAT"] = np.nan
boston_missing.loc[[4, 8], "PTRATIO"] = np.nan

boston_missing.head(10)

In [ ]:
boston_missing[["RM", "LSTAT", "PTRATIO"]].isnull().sum()

In [ ]:
# 평균 또는 중앙값으로 결측값 채우기
boston_filled = boston_missing.copy()

boston_filled["RM"] = boston_filled["RM"].fillna(boston_filled["RM"].mean())
boston_filled["LSTAT"] = boston_filled["LSTAT"].fillna(boston_filled["LSTAT"].median())
boston_filled["PTRATIO"] = boston_filled["PTRATIO"].fillna(boston_filled["PTRATIO"].mean())

boston_filled[["RM", "LSTAT", "PTRATIO"]].isnull().sum()

## 19. 실제 데이터 기술통계

In [ ]:
boston.describe().T

In [ ]:
# 관심 변수만 요약
boston[["Price", "RM", "LSTAT", "PTRATIO", "NOX", "DIS", "TAX"]].agg(["count", "mean", "median", "std", "var", "min", "max"])

In [ ]:
# 분위수 확인
boston[["Price", "RM", "LSTAT"]].quantile([0.25, 0.5, 0.75, 0.9])

## 20. apply를 실제 데이터에 적용하기

In [ ]:
# 컬럼별 범위 계산
boston[["Price", "RM", "LSTAT", "PTRATIO", "NOX", "DIS"]].apply(lambda x: x.max() - x.min())

In [ ]:
# 행 단위 적용: 간단한 가격 구간 라벨 만들기
price_q1 = boston["Price"].quantile(0.25)
price_q3 = boston["Price"].quantile(0.75)

def price_group(row):
    if row["Price"] >= price_q3:
        return "상위가격군"
    elif row["Price"] <= price_q1:
        return "하위가격군"
    else:
        return "중간가격군"

boston_labeled = boston.copy()
boston_labeled["가격구간"] = boston_labeled.apply(price_group, axis=1)
boston_labeled[["Price", "RM", "LSTAT", "가격구간"]].head(10)

In [ ]:
boston_labeled["가격구간"].value_counts()

## 21. 정렬과 순위 실습

In [ ]:
# 가격이 높은 순서로 정렬
boston_labeled.sort_values("Price", ascending=False).head(10)

In [ ]:
# Price 순위 만들기
boston_labeled["Price_rank"] = boston_labeled["Price"].rank(ascending=False, method="min")
boston_labeled[["Price", "RM", "LSTAT", "Price_rank"]].sort_values("Price_rank").head(10)

---

# Part 8. 상관관계와 공분산

- **공분산(covariance)**: 두 변수가 함께 변하는 방향과 정도를 나타냅니다. 단위의 영향을 받습니다.
- **상관계수(correlation coefficient)**: 두 변수의 선형 관계 강도를 -1에서 1 사이 값으로 표준화한 지표입니다.

Pandas에서는 다음 메서드를 자주 사용합니다.

- `df.cov()`
- `df.corr()`
- `df.corrwith()`

## 22. 공분산 계산

In [ ]:
focus_cols = ["Price", "RM", "LSTAT", "PTRATIO", "NOX", "DIS", "TAX"]

cov_matrix = boston[focus_cols].cov()
cov_matrix

## 23. 상관계수 계산

In [ ]:
corr_matrix = boston[focus_cols].corr()
corr_matrix

In [ ]:
# Price와 다른 변수의 상관관계
price_corr = boston[focus_cols].corrwith(boston["Price"]).sort_values(ascending=False)
price_corr

## 24. 상관관계 해석 실습

아래 결과에서 `Price`와 양의 상관관계가 큰 변수, 음의 상관관계가 큰 변수를 찾아봅니다.

In [ ]:
price_corr_without_self = price_corr.drop("Price")

print("Price와 가장 양의 상관관계가 큰 변수")
display(price_corr_without_self.sort_values(ascending=False).head(3))

print("Price와 가장 음의 상관관계가 큰 변수")
display(price_corr_without_self.sort_values(ascending=True).head(3))

## 25. 상관관계 시각화

Matplotlib만 사용해서 간단한 상관계수 행렬 이미지를 그립니다.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix)

ax.set_xticks(range(len(focus_cols)))
ax.set_yticks(range(len(focus_cols)))
ax.set_xticklabels(focus_cols, rotation=45, ha="right")
ax.set_yticklabels(focus_cols)

for i in range(len(focus_cols)):
    for j in range(len(focus_cols)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=ax)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.show()

---

# Part 9. 종합 실습 문제

아래 문제는 강사가 설명 후 학습자가 직접 완성할 수 있도록 구성했습니다.

## 문제 1. 직접 만든 DataFrame 분석

`students_filled`를 사용하여 다음을 수행해 보세요.

1. `반`별 평균점수를 계산하세요.
2. `평가등급`별 인원 수를 계산하세요.
3. 평균점수가 80점 이상이고 출석률이 0.9 이상인 학습자만 선택하세요.
4. `중간점수`, `기말점수`, `프로젝트점수` 사이의 상관계수를 계산하세요.

In [ ]:
# 문제 1 풀이 예시
print("1. 반별 평균점수")
display(students_filled.groupby("반")["평균점수"].mean())

print("2. 평가등급별 인원 수")
display(students_filled["평가등급"].value_counts())

print("3. 평균점수 80 이상, 출석률 0.9 이상")
display(students_filled[(students_filled["평균점수"] >= 80) & (students_filled["출석률"] >= 0.9)])

print("4. 점수 컬럼 간 상관계수")
display(students_filled[["중간점수", "기말점수", "프로젝트점수"]].corr())

## 문제 2. boston.csv 분석

`boston`을 사용하여 다음을 수행해 보세요.

1. `Price`가 상위 10%인 행만 선택하세요.
2. `RM` 기준 내림차순으로 정렬하세요.
3. `Price`, `RM`, `LSTAT`, `PTRATIO`의 평균, 표준편차, 중앙값을 계산하세요.
4. `Price`와 가장 상관관계가 큰 변수 5개를 찾으세요.

In [ ]:
# 문제 2 풀이 예시
print("1. Price 상위 10%")
threshold = boston["Price"].quantile(0.9)
display(boston[boston["Price"] >= threshold].head())

print("2. RM 기준 내림차순")
display(boston.sort_values("RM", ascending=False).head())

print("3. 관심 변수 요약")
display(boston[["Price", "RM", "LSTAT", "PTRATIO"]].agg(["mean", "std", "median"]))

print("4. Price와 상관관계가 큰 변수 5개")
display(boston.corr(numeric_only=True)["Price"].drop("Price").abs().sort_values(ascending=False).head(5))

---

# 마무리 정리

이번 실습에서 다룬 핵심 내용은 다음과 같습니다.

- Series는 1차원 라벨 데이터 구조입니다.
- DataFrame은 행과 열을 가진 2차원 표 형태 데이터 구조입니다.
- `index`, `columns`, `values`, `dtypes`, `shape`, `ndim`, `size`로 구조를 확인할 수 있습니다.
- `loc`는 라벨 기반, `iloc`는 정수 위치 기반 선택에 사용합니다.
- 결측값은 `isnull`, `notnull`, `dropna`, `fillna`로 확인하고 처리할 수 있습니다.
- `apply`는 행 또는 열 단위로 사용자 정의 함수를 적용할 때 유용합니다.
- `sort_index`, `sort_values`, `rank`로 정렬과 순위를 다룰 수 있습니다.
- `describe`, `mean`, `median`, `std`, `var`, `quantile`로 기술통계를 계산합니다.
- `corr`, `cov`, `corrwith`로 상관관계와 공분산을 분석합니다.